# RetailPulse: Data Cleaning and Preparation

## Overview

This notebook focuses on cleaning and preparing the Amazon Sale Report dataset for reliable analysis. It applies data quality findings from the previous notebook to address missing values, inconsistent categorical data, data types, and other identified issues.

## Objectives

* Create a working copy of the raw dataset for cleaning.
* Remove columns identified as unnecessary for the planned analysis.
* Handle missing values using appropriate strategies.
* Standardize categorical values and correct data types where needed.
* Investigate unusual values and validate the cleaned dataset.
* Save the cleaned dataset for subsequent analysis.

## Approach

The original raw dataset will remain unchanged. All cleaning operations will be performed on a separate working DataFrame, and the results will be validated before saving the processed dataset.


## 1. Load the Dataset

The raw Amazon Sale Report dataset is loaded into a Pandas DataFrame. A separate working copy is created to ensure that all cleaning operations leave the original dataset unchanged.


In [3]:
import pandas as pd
import numpy as np

csv_file = "../data/raw/Amazon Sale Report.csv"

df = pd.read_csv(csv_file)

df_cleaned = df.copy()

print("Dataset loaded successfully.")
print(f"Original dataset shape: {df.shape}")
print(f"Working copy shape: {df_cleaned.shape}")

Dataset loaded successfully.
Original dataset shape: (128975, 24)
Working copy shape: (128975, 24)


C:\Users\praya\AppData\Local\Temp\ipykernel_12844\4064105777.py:6: DtypeWarning: Columns (23: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


## 2. Standardize Column Names

This section standardizes column names by removing leading and trailing whitespace and replacing spaces with underscores. This creates consistent, easy-to-reference column names for subsequent cleaning and analysis.


In [8]:
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Columns names are standardized successfully.")
df_cleaned.columns.to_list()

Columns names are standardized successfully.


['index',
 'order_id',
 'date',
 'status',
 'fulfilment',
 'sales_channel',
 'ship_service_level',
 'style',
 'sku',
 'category',
 'size',
 'asin',
 'courier_status',
 'qty',
 'currency',
 'amount',
 'ship_city',
 'ship_state',
 'ship_postal_code',
 'ship_country',
 'promotion_ids',
 'b2b',
 'fulfilled_by',
 'unnamed:_22']

## 3. Remove Unnecessary Columns

This section removes columns identified during the data quality assessment as unnecessary for the planned analysis. The selected columns are removed from the working DataFrame, while the original raw dataset remains unchanged.


In [34]:
drop_columns = [
    "index",
    "currency",
    "ship_city",
    "ship_postal_code",
    "promotion_ids",
    "unnamed:_22"
]

drop_columns

['index',
 'currency',
 'ship_city',
 'ship_postal_code',
 'promotion_ids',
 'unnamed:_22']

In [43]:
# df_cleaned.drop(columns=drop_columns, inplace=True)

In [44]:
print(f"Columns removed: {len(drop_columns)}")
print(f"Remaining columns: {df_cleaned.shape[1]}")

df_cleaned.columns.to_list()

Columns removed: 6
Remaining columns: 18


['order_id',
 'date',
 'status',
 'fulfilment',
 'sales_channel',
 'ship_service_level',
 'style',
 'sku',
 'category',
 'size',
 'asin',
 'courier_status',
 'qty',
 'amount',
 'ship_state',
 'ship_country',
 'b2b',
 'fulfilled_by']

## 4. Investigate Duplicate Order IDs

An order ID may appear in multiple rows when an order contains multiple products or line items. Therefore, repeated order IDs do not necessarily indicate duplicate records.

This section identifies repeated order IDs and examines their frequency before deciding how to handle them. Valid line items will be preserved to avoid losing sales data.


In [ ]:
print("Duplicate order ids: ", df_cleaned["order_id"].duplicated().sum())

print("Exact duplicate records along dataset: ", df_cleaned.duplicated().sum())


Duplicate order ids:  8597
Exact duplicate records along dataset:  6


In [48]:
duplicates_before = df_cleaned.duplicated().sum()

df_cleaned.drop_duplicates(inplace=True)

duplicates_after = df_cleaned.duplicated().sum()

print(f"Exact duplicate records removed: {duplicates_before - duplicates_after}")
print(f"Exact duplicate records remaining: {duplicates_after}")
print(f"Updated dataset shape: {df_cleaned.shape}")

Exact duplicate records removed: 6
Exact duplicate records remaining: 0
Updated dataset shape: (128969, 18)


## 5. Convert the Date Column

The `date` column represents the date associated with each sales record. Converting it to a datetime data type enables accurate date-based filtering, time-series analysis, and the calculation of daily, monthly, and yearly sales trends.

This section converts the date values to a datetime format and checks for any values that could not be converted successfully.


In [66]:
df_cleaned["date"] = pd.to_datetime(
    df_cleaned["date"],
    format="%m-%d-%y",
    errors="coerce"
)

print("Date column converted successfully.")
print(f"Data type: {df_cleaned['date'].dtype}")
print(f"Missing dates: {df_cleaned['date'].isna().sum()}")
print(f"Date range: {df_cleaned['date'].min()} to {df_cleaned['date'].max()}")

Date column converted successfully.
Data type: datetime64[us]
Missing dates: 0
Date range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00
